## LLM Zoomcamp
### Homework 2

Homework questions are at https://github.com/DataTalksClub/llm-zoomcamp/blob/main/cohorts/2026/02-vector-search/homework.md

First, let's download these two helper scripts in our working directory:

- https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/embed/download.py
- https://github.com/DataTalksClub/llm-zoomcamp/blob/main/02-vector-search/embed/embedder.py


In [1]:
# By default download.py fetches Xenova/all-MiniLM-L6-v2, the ONNX version of the all-MiniLM-L6-v2 model from the lessons.
#%run download.py

  saved models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  saved models/Xenova/all-MiniLM-L6-v2/model.onnx


In [2]:
import sys
print(sys.version)

3.10.18 (main, Jun  5 2025, 08:13:51) [Clang 14.0.6 ]


## Q1. Embedding a query

In [3]:
from embedder import Embedder

In [4]:
query="How does approximate nearest neighbor search work?"

In [5]:
# Create a Embedder instance:
embedder = Embedder()

In [6]:
v=embedder.encode(query)

In [7]:
len(v)

384

In [8]:
print(v[0])

-0.02058200593003704


In [9]:
# Load data
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [10]:
len(documents) # Each document is a dictionary with a filename and content, and there are 72 pages.

72

## Q2. Cosine similarity

In [11]:
# Get content of the page 02-vector-search/lessons/07-sqlitesearch-vector.md
content= [document.get("content") for document in documents if document["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"]

In [12]:
print(content[0])

# Vector Search with sqlitesearch

Video: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)

In the previous section we used minsearch for vector search.

It works, but it has three problems:

- It rebuilds the index on every startup
- It keeps everything in memory
- It searches by brute force


With text search we never felt these. Indexing was fast because we
didn't embed anything. With vector search, indexing runs a neural
network over every document, so it takes a minute on our dataset.
Keeping everything in memory is fine here, but a larger dataset would
need too much space.

The third problem is brute-force search. For every query we compare the
query vector against every single document. With 1,000 documents this is
fine, probably even faster than anything smarter. But as the dataset
grows past 10,000 or so, it gets slow, and we'll want an approximate
method instead.

What we've done so far is exact nearest neighbor (NN) sea

In [13]:
# Embed the content
c=embedder.encode(content[0])

In [14]:
type(c)

numpy.ndarray

In [15]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


In [16]:
# Reshape vectors to 2D arrays (1, -1) as required by sklearn
v_2d = v.reshape(1, -1)
c_2d = c.reshape(1, -1)

In [17]:
similarity = cosine_similarity(v_2d, c_2d)
print(f"Cosine Similarity: {similarity[0][0]}")

Cosine Similarity: 0.3610702814461231


## Q3. Chunking and search by hand

In [18]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)

In [19]:
len(chunks)

295

In [20]:
chunk_contents = [chunk["content"] for chunk in chunks]

In [21]:
chunk_contents[0:2]

['# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simple language

In [22]:
# We embed every chunk's content with encode_batch, stack the vectors into a matrix X, and score the Q1 query against all chunks:

vectors=embedder.encode_batch(chunk_contents)

In [23]:
len(vectors)

295

In [24]:
X = np.array(vectors)

In [25]:
X.shape

(295, 384)

In [26]:
scores = X.dot(v)

In [27]:
# Which file does the highest-scoring chunk belong to (its filename)?
chunks[np.argmax(scores)]["filename"]

'02-vector-search/lessons/07-sqlitesearch-vector.md'

## Q4. Vector search with minsearch

In [28]:
from minsearch import VectorSearch

In [29]:
query="What metric do we use to evaluate a search engine?"

In [30]:
vindex = VectorSearch(keyword_fields=["content"])
vindex.fit(X, chunks)

In [31]:
query_vector = embedder.encode(query)
results = vindex.search(query_vector, num_results=5)

In [32]:
# Top result's filename:
results[0]['filename']

'04-evaluation/lessons/05-search-metrics.md'

## Q5. Text search vs vector search

In [33]:
query='How do I store vectors in PostgreSQL?'

In [34]:
# Vector search
query_vector = embedder.encode(query)
results_vector_search = vindex.search(query_vector, num_results=5)

In [35]:
#Text search
from minsearch import Index
chunks_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunks_index.fit(chunks)
results_text_search = chunks_index.search(query, num_results=5)

In [36]:
[result["filename"] for result in results_vector_search]

['02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/08-pgvector.md',
 '02-vector-search/lessons/08-pgvector.md']

In [37]:
[result["filename"] for result in results_text_search]

['02-vector-search/lessons/02-embeddings.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md',
 '03-orchestration/lessons/05-rag.md',
 '02-vector-search/lessons/01-intro.md']

## Q6. Hybrid search

In [38]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [39]:
query = "How do I give the model access to tools?"

In [40]:
# Vector search
query_vector = embedder.encode(query)
results_vector_search = vindex.search(query_vector)

In [41]:
#Text search
from minsearch import Index
chunks_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunks_index.fit(chunks)
results_text_search = chunks_index.search(query)

In [42]:
results = rrf([results_vector_search, results_text_search])

In [43]:
print(results[0]['filename'])

01-agentic-rag/lessons/13-function-calling.md
